# PDF → Qdrant 멀티모달 적재 파이프라인 (OpenAI 버전)

PDF 안의 **본문 텍스트 / 표 / 이미지**를 각각 다르게 처리해서 Qdrant 벡터DB에 적재하는 파이프라인입니다.
캡션/요약 생성과 임베딩 모두 **OpenAI** 모델을 사용합니다.

전체 흐름:
1. PDF에서 텍스트 블록 추출 (PyMuPDF)
2. PDF에서 표 추출 → 마크다운 변환 (pdfplumber)
3. PDF에서 이미지 추출 (PyMuPDF)
4. 이미지 → GPT-4o Vision으로 캡션(설명) 생성
5. 표 → GPT-4o로 요약 생성
6. 텍스트/표/이미지를 타입별 메타데이터를 붙여 청크로 통합
7. 청크를 OpenAI Embeddings로 임베딩
8. Qdrant collection 생성 후 업서트
9. 검색 예시 (타입 필터 포함)

> **필요한 것**: `OPENAI_API_KEY` 환경변수, 로컬 또는 원격 Qdrant 인스턴스


## 0. 패키지 설치

## 1. 설정 및 기본 임포트

In [74]:
import os
import io
import base64
import uuid
from pathlib import Path

import fitz  # PyMuPDF
import pdfplumber
from PIL import Image
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue

# ----- 설정값: 필요에 맞게 수정하세요 -----
#pdf 두개 :
PDF_PATH = "길라잡이/대한민국+국방부복지정책과,+2026년+병+복지+길라잡이,+2026.+3.pdf"          # 처리할 PDF 경로

QDRANT_URL = "http://localhost:6333"   # Qdrant 서버 주소 (Qdrant Cloud면 해당 URL로 교체)
QDRANT_API_KEY = None                    # Qdrant Cloud 쓰면 API 키 입력
COLLECTION_NAME = "GUIDANCE"

CHAT_MODEL = "gpt-4o"                          # 캡션/요약용 모델 (vision 지원)
EMBED_MODEL = "text-embedding-3-large"         # OpenAI 임베딩 모델 (다국어/한국어 지원 우수)
EMBED_DIM = 3072                                # text-embedding-3-large 기본 차원 (small은 1536)

# OpenAI 클라이언트 (환경변수 OPENAI_API_KEY 필요)
openai_client = OpenAI()


In [75]:
doc = fitz.open(PDF_PATH)
print(len(doc))

80


## 2. 본문 텍스트 블록 추출

PyMuPDF로 페이지별 텍스트 블록을 뽑습니다. 표/이미지 영역과 겹치는 텍스트는 이후 단계에서 자연히 분리되므로,
여기서는 우선 페이지 단위 순수 텍스트만 뽑고 문단 단위로 나눕니다.

In [76]:
def extract_text_blocks(pdf_path: str):
    """페이지별로 텍스트 블록을 추출한다. 각 블록은 dict(page, text, bbox)"""
    doc = fitz.open(pdf_path)
    blocks_out = []
    for page_num, page in enumerate(doc, start=1):
        blocks = page.get_text("blocks")  # (x0, y0, x1, y1, text, block_no, block_type)
        for b in blocks:
            text = b[4].strip()
            if not text:
                continue
            blocks_out.append({
                "page": page_num,
                "text": text,
            })
    doc.close()
    return blocks_out


In [77]:
extract_text_blocks(PDF_PATH)

[{'page': 1, 'text': '길라잡이\n길라잡이'},
 {'page': 1, 'text': '병 복지'},
 {'page': 1, 'text': '병 복지혜택 이렇게 다양합니다\n2026'},
 {'page': 2, 'text': 'C\nO\nN\nT\nE\nN\nT\nS'},
 {'page': 2, 'text': '휴가··································· 42'},
 {'page': 2, 'text': '유자녀 현역병의 상근예비역 편입·········· 44'},
 {'page': 2, 'text': '전역··································· 45'},
 {'page': 2, 'text': '가사사정으로 인한 전역·················· 45'},
 {'page': 2, 'text': 'Part 03'},
 {'page': 2, 'text': '인사제도 /\n  근무 분야'},
 {'page': 2, 'text': '군 복무간 중단 없는 학습기회 제공· ······· 30'},
 {'page': 2, 'text': '국가기술자격 취득······················· 36\nPart 02'},
 {'page': 2, 'text': '자기개발 분야'},
 {'page': 2, 'text': '보수···································· 6'},
 {'page': 2, 'text': '급식 및 물자····························· 9'},
 {'page': 2, 'text': '병영생활관 개선 ························ 10'},
 {'page': 2, 'text': '현역병 학자금 대출이자 면제·············· 10'},
 {'page': 2, 'text': '나라사랑카드 ··························· 11'},
 {'page': 2, 'text': '나라사랑포털 운영·············

## 3. 표 추출 (→ 마크다운 변환)

pdfplumber는 표의 셀 구조를 잘 잡아줍니다. 추출한 표는 마크다운 표 형식 문자열로 변환해서 저장합니다.
원본 텍스트 형태의 표 데이터는 나중에 payload에 그대로 보존해서, 검색 후 LLM에 넘길 때 정확한 표를 그대로 쓸 수 있게 합니다.

In [78]:
def table_to_markdown(table_rows):
    """pdfplumber의 extract_table() 결과(2차원 리스트)를 마크다운 표로 변환"""
    rows = [[cell if cell is not None else "" for cell in row] for row in table_rows]
    if not rows:
        return ""
    header = rows[0]
    body = rows[1:]
    md_lines = []
    md_lines.append("| " + " | ".join(header) + " |")
    md_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    for row in body:
        md_lines.append("| " + " | ".join(row) + " |")
    return "\n".join(md_lines)


def extract_tables(pdf_path: str):
    """페이지별로 표를 추출해 마크다운 문자열 리스트로 반환"""
    tables_out = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                if not table or len(table) < 2:
                    continue
                md_table = table_to_markdown(table)
                tables_out.append({
                    "page": page_num,
                    "markdown": md_table,
                })
    return tables_out


## 6. 표 요약 생성 (GPT-4o)

표 자체를 임베딩하기보다, 표가 무엇을 말하는지 요약한 텍스트를 임베딩용으로 쓰고
원본 마크다운 표는 payload에 그대로 보존합니다.

In [79]:
def summarize_table(table_markdown: str) -> str:
    """표 마크다운을 GPT-4o에 넘겨 검색용 요약 텍스트 생성"""
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": (
                "다음은 문서에서 추출한 표입니다. 이 표가 어떤 내용을 담고 있는지 "
                "검색에 활용할 수 있도록 2~4문장의 한국어로 요약해줘. "
                "컬럼이 의미하는 바와 주요 수치/항목을 포함해줘.\n\n"
                f"{table_markdown}"
            ),
        }],
    )
    return response.choices[0].message.content.strip()


## 7. 청크 통합

텍스트/표/이미지를 하나의 통일된 청크 리스트로 합칩니다.
각 청크는 `text`(임베딩 대상), `type`(text/table/image), 그리고 원본 데이터를 담은 `payload_extra`를 가집니다.

In [80]:
def chunk_text_blocks(text_blocks, max_chars=800):
    """페이지 텍스트 블록들을 적당한 크기로 합쳐서 청크로 만든다"""
    chunks = []
    buffer = ""
    buffer_page = None
    for b in text_blocks:
        if buffer_page is None:
            buffer_page = b["page"]
        # 페이지가 바뀌거나 버퍼가 너무 커지면 flush
        if b["page"] != buffer_page or len(buffer) + len(b["text"]) > max_chars:
            if buffer.strip():
                chunks.append({"page": buffer_page, "text": buffer.strip()})
            buffer = ""
            buffer_page = b["page"]
        buffer += b["text"] + "\n"
    if buffer.strip():
        chunks.append({"page": buffer_page, "text": buffer.strip()})
    return chunks


def build_chunks(pdf_path: str):
    print("1) 텍스트 블록 추출 중...")
    text_blocks = extract_text_blocks(pdf_path)
    text_chunks = chunk_text_blocks(text_blocks)

    print("2) 표 추출 중...")
    tables = extract_tables(pdf_path)


    all_chunks = []

    for c in text_chunks:
        all_chunks.append({
            "type": "text",
            "page": c["page"],
            "embed_text": c["text"],
            "payload_extra": {"content": c["text"]},
        })

    print(f"4) 표 {len(tables)}개 요약 생성 중...")
    for t in tables:
        summary = summarize_table(t["markdown"])
        all_chunks.append({
            "type": "table",
            "page": t["page"],
            "embed_text": summary,
            "payload_extra": {"table_markdown": t["markdown"], "summary": summary},
        })


    print(f"완료: 텍스트 {len(text_chunks)} / 표 {len(tables)} → 총 {len(all_chunks)} 청크")
    return all_chunks


## 8. 임베딩 (OpenAI Embeddings)

`text-embedding-3-large` 모델을 사용합니다. 한국어 포함 다국어 성능이 좋습니다.
필요하면 비용 절감을 위해 `text-embedding-3-small`(1536차원)로 바꿔도 됩니다 — 이 경우 `EMBED_DIM`도 1536으로 수정하세요.

In [81]:
def embed_texts(texts, batch_size: int = 100):
    """OpenAI Embeddings API로 텍스트 리스트를 임베딩. 배치 단위로 호출."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch)
        # 응답 순서는 입력 순서와 동일하게 보장됨
        all_embeddings.extend([d.embedding for d in response.data])
        print(f"임베딩 진행: {min(i + batch_size, len(texts))}/{len(texts)}")
    return all_embeddings


## 9. Qdrant collection 생성 및 업서트

In [82]:
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def ensure_collection(vector_size: int):
    existing = [c.name for c in qdrant.get_collections().collections]
    if COLLECTION_NAME in existing:
        print(f"'{COLLECTION_NAME}' collection이 이미 존재합니다.")
        return
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )
    print(f"'{COLLECTION_NAME}' collection 생성 완료 (dim={vector_size})")


def upsert_chunks(pdf_path: str, chunks, embeddings):
    doc_name = Path(pdf_path).name
    points = []
    for chunk, vector in zip(chunks, embeddings):
        payload = {
            "doc_name": doc_name,
            "page": chunk["page"],
            "type": chunk["type"],
            "text": chunk["embed_text"],
            **chunk["payload_extra"],
        }
        points.append(PointStruct(id=str(uuid.uuid4()), vector=vector, payload=payload))

    qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"{len(points)}개 포인트 업서트 완료")


## 10. 전체 파이프라인 실행

In [83]:
def run_pipeline(pdf_path: str):
    chunks = build_chunks(pdf_path)

    print("6) 임베딩 생성 중...")
    texts = [c["embed_text"] for c in chunks]
    embeddings = embed_texts(texts)

    ensure_collection(vector_size=len(embeddings[0]))

    print("7) Qdrant 업서트 중...")
    upsert_chunks(pdf_path, chunks, embeddings)

    print("파이프라인 완료!")
    return chunks


# 실행 예시
# chunks = run_pipeline(PDF_PATH)


In [84]:
chunks = run_pipeline(PDF_PATH)

1) 텍스트 블록 추출 중...
2) 표 추출 중...
4) 표 59개 요약 생성 중...
완료: 텍스트 115 / 표 59 → 총 174 청크
6) 임베딩 생성 중...
임베딩 진행: 100/174
임베딩 진행: 174/174
'GUIDANCE' collection이 이미 존재합니다.
7) Qdrant 업서트 중...
174개 포인트 업서트 완료
파이프라인 완료!


## 11. 검색 예시

일반 검색과, `type` 필터로 표/이미지만 찾는 예시입니다.

In [72]:
def search(query: str, top_k: int = 5, type_filter: str = None):
    query_vector = embed_texts([query])[0]

    qfilter = None
    if type_filter:
        qfilter = Filter(must=[FieldCondition(key="type", match=MatchValue(value=type_filter))])

    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=top_k,
        query_filter=qfilter,
    )

    for r in results:
        print(f"[score={r.score:.3f}] page={r.payload['page']} type={r.payload['type']}")
        print(r.payload["text"][:200])
        print("-" * 40)

    return results


# 사용 예시:
# search("2023년 매출 추이")
# search("매출 그래프", type_filter="image")
# search("분기별 실적표", type_filter="table")


In [73]:
search("분기별 실적표", type_filter="table")

임베딩 진행: 1/1


AttributeError: 'QdrantClient' object has no attribute 'search'